# Treino do modelo PRÓPRIO de EPI (capacete + colete) — sem Roboflow

Usa o dataset público **Construction-PPE** da Ultralytics (1.416 fotos de obra com `Person`, `helmet`, `no_helmet`, `vest`...),
baixado direto do GitHub da Ultralytics — **não precisa de conta nem de chave**.

Rode as células **uma por vez, de cima para baixo** (botão ▶ à esquerda de cada uma).

Antes de começar: menu **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU → Salvar**.

In [ ]:
# PASSO 1 - Verifica a GPU e instala o YOLO
import torch
print('GPU disponível:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NÃO! Ative a T4 GPU no menu e rode de novo.')
!pip install -q ultralytics

In [ ]:
# PASSO 2 - Baixa as fotos de treino (178 MB, direto do GitHub da Ultralytics, sem conta)
import glob, os
from ultralytics.data.utils import check_det_dataset

dados = check_det_dataset('construction-ppe.yaml')  # baixa e descompacta na primeira vez
print('\nClasses:', list(dados['names'].values()))
for parte in ('train', 'val', 'test'):
    print(f'{parte}: {len(os.listdir(dados[parte]))} imagens')
faltam = [c for c in ('Person', 'helmet', 'no_helmet', 'vest') if c not in dados['names'].values()]
print('OK, dataset pronto!' if not faltam else f'ATENÇÃO: faltam as classes {faltam}')

In [ ]:
# PASSO 3 - Treina o SEU modelo (uns 30 a 50 minutos). Pode deixar rodando, mas NÃO feche a aba.
# 'yolo11s' = bom equilíbrio entre precisão e velocidade para rodar depois num PC sem placa de vídeo.
from ultralytics import YOLO
modelo = YOLO('yolo11s.pt')
modelo.train(data='construction-ppe.yaml', epochs=80, imgsz=640, patience=20,
             project='/content/treino', name='epi', exist_ok=True)

In [ ]:
# PASSO 4 - Resultado: precisão por classe (olhe a coluna mAP50 de Person, helmet, no_helmet e vest)
melhor = YOLO('/content/treino/epi/weights/best.pt')
metricas = melhor.val(data='construction-ppe.yaml', split='test')
print(f'\n>>> mAP50 geral: {metricas.box.map50:.2f}')

# Mostra uma imagem de teste com as detecções
from IPython.display import Image, display
imagem_teste = sorted(glob.glob(os.path.join(dados['test'], '*')))[0]
melhor.predict(imagem_teste, save=True, project='/content/teste', name='img', exist_ok=True)
display(Image(glob.glob('/content/teste/img/*')[0], width=700))

In [ ]:
# PASSO 5 - Baixa o modelo pronto (epi.pt) para o seu computador
import shutil
from google.colab import files
shutil.copy('/content/treino/epi/weights/best.pt', '/content/epi.pt')
files.download('/content/epi.pt')

## Pronto!
Me avise quando o `epi.pt` estiver na sua pasta Downloads — eu coloco no lugar e troco o motor para `local`.

Depois disso o sistema roda **100% no seu PC**: sem Roboflow, sem internet e sem custo por análise.